# XiDepth v2.0 — Track 1: ResNet-18 Baseline, Parity Gate & Step Timing

This notebook executes the rigorous scientific validation workflow in order:
1. **Prerequisites & Verification**: Checks GPU and attaches `awsaf49/kitti-eigen-split`.
2. **Dataset Integrity Gate**: Runs `verify_dataset_paths.py` across all train/val/test frames.
3. **Official Monodepth2 Parity Gate**: Evaluates official weights on 652 benchmark (Requires AbsRel ~0.090).
4. **Empirical Step Timing**: Times 100 actual training steps with full dataloader overhead to measure true step latency.
5. **Track 1 Baseline Training**: Launches ResNet-18 self-supervised training calibrated to the measured budget.
6. **Track 1b Control Training**: Launches ResNet-18 from-scratch control run.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
# 2. Locate KITTI dataset on Kaggle
import os, glob
input_dirs = glob.glob("/kaggle/input/*kitti*")
print("Found candidate input dirs:", input_dirs)
dataset_dir = input_dirs[0] if input_dirs else "/kaggle/input/kitti-eigen-split"
print(f"Using dataset directory: {dataset_dir}")


In [ ]:
# 3. Clone or pull XiDepth-Monocular-2.0 and install dependencies
!git clone https://github.com/ASMITSARKAR/XiDepth-Monocular-2.0.git xidepth_repo
%cd xidepth_repo
!pip install -r requirements.txt

# Ensure official weights and improved GT are present
!mkdir -p checkpoints/monodepth2_official data
!wget -q -nc https://storage.googleapis.com/niantic-lon-models/monodepth2/mono_640x192.zip -O checkpoints/monodepth2_official/mono_640x192.zip
!unzip -o -q checkpoints/monodepth2_official/mono_640x192.zip -d checkpoints/monodepth2_official/
!wget -q -nc "https://www.dropbox.com/scl/fi/dg7eskv5ztgdyp4ippqoa/gt_depths.npz?rlkey=qb39aajkbhmnod71rm32136ry&dl=1" -O data/gt_depths.npz


In [ ]:
# 4. Gate 1: Verify all dataset paths and calib files exist on disk (Must Exit 0)
!python scripts/verify_dataset_paths.py --dataset_dir "$dataset_dir"


In [ ]:
# 5. Gate 2: Run Official Monodepth2 Parity Benchmark Test (Must hit AbsRel ~0.090)
!python scripts/eval.py \
    --model monodepth2_official \
    --checkpoint checkpoints/monodepth2_official \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_benchmark/test_files.txt \
    --gt_path data/gt_depths.npz \
    --output_json /kaggle/working/monodepth2_parity_results.json


In [ ]:
# 6. Step Timing: Time exactly 100 training steps (1,200 samples @ batch 12)
# Measures real data-loader and forward/backward GPU latency before starting full run
!python scripts/train.py \
    --model resnet18 \
    --pretrained \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_timing \
    --batch_size 12 \
    --epochs 1 \
    --max_samples 1200 \
    --use_amp \
    --log_freq 20


In [ ]:
# 7. Launch Track 1 Baseline Training (ResNet-18 ImageNet Pretrained)
!python scripts/train.py \
    --model resnet18 \
    --pretrained \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_resnet18 \
    --batch_size 12 \
    --epochs 20 \
    --lr 1e-4 \
    --lr_step_size 15 \
    --num_workers 4 \
    --use_amp \
    --log_freq 100


In [ ]:
# 8. Evaluate Track 1 Pretrained Model on Eigen Benchmark (652 files)
!python scripts/eval.py \
    --model resnet18 \
    --checkpoint /kaggle/working/checkpoints_resnet18/best_model.pth \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_benchmark/test_files.txt \
    --gt_path data/gt_depths.npz \
    --output_json /kaggle/working/resnet18_pretrained_results.json


In [ ]:
# 9. Launch Track 1b Control Training (ResNet-18 From-Scratch)
!python scripts/train.py \
    --model resnet18 \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_resnet18_scratch \
    --batch_size 12 \
    --epochs 20 \
    --lr 1e-4 \
    --lr_step_size 15 \
    --num_workers 4 \
    --use_amp \
    --log_freq 100


In [ ]:
# 10. Evaluate Track 1b From-Scratch Control Model on Eigen Benchmark (652 files)
!python scripts/eval.py \
    --model resnet18 \
    --checkpoint /kaggle/working/checkpoints_resnet18_scratch/best_model.pth \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_benchmark/test_files.txt \
    --gt_path data/gt_depths.npz \
    --output_json /kaggle/working/resnet18_scratch_results.json
